# Week 3 recap — Sheet 03 SOLUTIONS: numbers you must not average

Executed in the lab image. Every quoted number is what it actually printed.

Question 4 is the one that matters in a meeting: two defensible methods, a real
gap between them, and no way to tell from the output which one you are looking
at.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Week 3 recap, sheet 03 — Numbers you must not average. Run this once.
import glob
import pandas as pd

BRONZE = "data/bronze/"
orders = (pd.concat([pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))],
                    ignore_index=True)
            .drop_duplicates(subset="LineID"))
customers = pd.read_csv(BRONZE + "customers.csv")
returns = pd.read_csv(BRONZE + "returns.csv")

sales = orders.merge(customers[["CustomerID", "Region", "CustomerSegment"]],
                     on="CustomerID", how="left", validate="many_to_one")
sales["IsReturned"] = sales.OrderID.isin(set(returns.OrderID))
sales["ReturnedSales"] = sales.Sales.where(sales.IsReturned, 0.0)

print("sales:", sales.shape)
print("regions:", sales.Region.nunique(), "| segments:", sales.CustomerSegment.nunique())

PART A — the average of averages

### Question 1

Compute mean `Sales` per `Region`. Then produce the overall average two ways: the mean of those regional means, and the mean over all rows. Print both and the gap.
> **NOTE:** day 2 worksheet 05 found `1995.2` against `1522.22` doing exactly this.

In [ ]:
per_region = sales.groupby("Region").Sales.agg(["size", "mean"])
print(per_region.round(2).to_string())
print()
unweighted = per_region["mean"].mean()
weighted = sales.Sales.mean()
print("mean of the %d regional means: %10.2f" % (len(per_region), unweighted))
print("mean over all %d rows:        %10.2f" % (len(sales), weighted))
print("gap:                          %10.2f  (%.1f%%)"
      % (unweighted - weighted, 100 * (unweighted - weighted) / weighted))

```
                       size     mean
Region
Atlantic               1038  1766.86
Northwest Territories   380  2057.09
Nunavut                  77  1339.46
Ontario                1741  1568.91
Prarie                 1640  1597.04
Quebec                  749  1773.86
West                   1921  1724.14
Yukon                   514  1674.43

mean of the 8 regional means:    1687.72
mean over all 8060 rows:         1683.72
gap:                                4.00  (0.2%)
```

Two averages of the same column, differing by **4.00**.

The gap is small here — 0.2% — and that is worth being honest about: day 2
worksheet 05 found `1995.2` against `1522.22`, a 31% gap, on data with a much
more lopsided distribution. The size of the error depends entirely on how uneven
the groups are, which question 2 measures.

But the *size* is not the point. Both numbers are printed by one-line pandas
calls, both look like "the average sale", and **nothing in the output says which
one you are looking at**. A colleague who recomputes your figure the other way
gets 1687.72 instead of 1683.72 and concludes your report is broken.

The mechanism: `per_region["mean"].mean()` averages eight numbers, giving each
region one vote. `sales.Sales.mean()` averages 8,060 rows, giving each *order
line* one vote. Nunavut has 77 lines and Ontario has 1,741, and the first method
treats those as equally important.

Which is right depends on the question. *"What does a typical order line look
like?"* → weighted. *"How do our regions compare on average order size?"* →
unweighted, and then say so.

### Question 2

Show why they differ. Print each region's row count as a share of the total, and the count of the largest and smallest regions.
> **NOTE:** an unweighted mean gives Nunavut the same vote as Ontario.

In [ ]:
per_region = sales.groupby("Region").Sales.agg(["size", "mean"])
per_region["share_pct"] = (100 * per_region["size"] / len(sales)).round(2)
print(per_region.sort_values("size", ascending=False).round(2).to_string())
print()
print("largest region: %d rows" % per_region["size"].max())
print("smallest region: %d rows" % per_region["size"].min())
print("ratio: %.0fx" % (per_region["size"].max() / per_region["size"].min()))

```
                       size     mean  share_pct
Region
West                   1921  1724.14      23.83
Ontario                1741  1568.91      21.60
Prarie                 1640  1597.04      20.35
Atlantic               1038  1766.86      12.88
Quebec                  749  1773.86       9.29
Yukon                   514  1674.43       6.38
Northwest Territories   380  2057.09       4.71
Nunavut                  77  1339.46       0.96

largest region: 1921 rows
smallest region: 77 rows
ratio: 25x
```

**25x between the largest and smallest region**, and Nunavut is **0.96%** of the
data.

That ratio is the whole explanation. In the unweighted mean, Nunavut's 77 lines
carry the same weight as West's 1,921 — a 25-fold overstatement of Nunavut's
influence on the headline figure.

And notice which regions are extreme. Northwest Territories has the *highest*
mean (2,057.09) on 4.71% of the rows; Nunavut has the *lowest* (1,339.46) on
0.96%. Both are small, both are outliers, and the unweighted mean gives them a
quarter of the vote for a
twentieth of the business.

This is why the gap is only 4.00 here: the two small extremes happen to pull in
opposite directions and largely cancel. That is luck. Remove Nunavut and the
unweighted mean jumps, for a region that is 1% of the data.

**The general rule: an average of group averages is only meaningful when the
groups are the same size, or when you genuinely mean to weight them equally.**
Neither is true by default, so state which you did.

Also worth noting: `Prarie` again — the misspelling recap sheet 02 found. It
propagates into every regional report, including this one.

PART B — rates

### Question 3

Compute the return rate by revenue for each `CustomerSegment`, two ways: the mean of per-row rates, and `SUM(ReturnedSales) / SUM(Sales)`. Print both columns side by side.

In [ ]:
g = sales.groupby("CustomerSegment").agg(
    lines=("LineID", "size"),
    returned=("ReturnedSales", "sum"),
    total=("Sales", "sum"))
g["ratio_of_sums"] = (g.returned / g.total).round(4)

per_row = sales.assign(rate=sales.ReturnedSales / sales.Sales)
g["mean_of_rates"] = per_row.groupby("CustomerSegment").rate.mean().round(4)

print(g[["lines", "ratio_of_sums", "mean_of_rates"]].to_string())
print()
print("overall ratio of sums: %.4f"
      % (sales.ReturnedSales.sum() / sales.Sales.sum()))
print("overall mean of rates: %.4f" % per_row.rate.mean())

```
                 lines  ratio_of_sums  mean_of_rates
CustomerSegment
Consumer          1584         0.0700         0.0890
Corporate         2947         0.1076         0.1099
Home Office       1952         0.1063         0.0973
Small Business    1577         0.1628         0.1154

overall ratio of sums: 0.1095
overall mean of rates: 0.1038
```

Two return rates per segment, and they disagree in **both directions**.

Consumer: 0.0700 by revenue, 0.0890 by line. Home Office: 0.1063 by revenue,
0.0973 by line. So this is not a constant bias you could correct for — the
direction depends on whether a segment's *expensive* lines or its *cheap* lines
are more likely to be returned.

Look at Small Business: **0.1628 against 0.1154**, the widest gap on the sheet.
By revenue it is by far the worst segment; by line count it is barely worse than
Corporate. Those two sentences would lead to different business decisions.

The overall figures differ too — 0.1095 against 0.1038 — and both are one-line
pandas calls that a reader would describe identically as "our return rate".

This is day 4 worksheet 03's `0.0875` vs `0.0835`, reproduced on different data.
It is not a rounding artefact and it is not a bug. It is the difference between
weighting by **dollars** and weighting by **rows**, and question 4 says which one
a business means.

### Question 4

Say which is correct. Print, for one segment, the total returned and total sales that produce the ratio — and explain in one sentence what the other method is actually measuring.
> **NOTE:** neither is wrong. They answer different questions, and the output does not say which you are looking at.

In [ ]:
seg = "Corporate"
sub = sales[sales.CustomerSegment == seg]
print("segment:", seg)
print("  lines:            %6d" % len(sub))
print("  SUM(Sales):    %12.2f" % sub.Sales.sum())
print("  SUM(Returned): %12.2f" % sub.ReturnedSales.sum())
print("  ratio of sums:      %.4f" % (sub.ReturnedSales.sum() / sub.Sales.sum()))
print()
rates = sub.ReturnedSales / sub.Sales
print("  mean of per-line rates: %.4f" % rates.mean())
print("  (each line counts once, whatever it is worth)")
print()
print("lines that are fully returned:", int((rates == 1).sum()))
print("lines not returned at all:    ", int((rates == 0).sum()))

```
segment: Corporate
  lines:              2947
  SUM(Sales):      5069948.54
  SUM(Returned):    545755.37
  ratio of sums:      0.1076

  mean of per-line rates: 0.1099
  (each line counts once, whatever it is worth)

lines that are fully returned: 324
lines not returned at all:     2623
```

**`ratio_of_sums` is the one to report**, and the two numbers underneath it are
why: 545,755.37 returned out of 5,069,948.54 sold. That division is a fact about
money, and both inputs are additive, so it re-aggregates correctly to any grain.

`mean_of_rates` is measuring something else. Because a line is either fully
returned or not at all — **324 fully returned, 2,623 not, and nothing in
between** — the per-line rate is a column of 1s and 0s. Its mean is therefore
just `324 / 2947`, the **proportion of lines** returned. A perfectly good metric,
answering a different question: *how often does a return happen*, not *how much
of our revenue comes back*.

So the two numbers are the **return frequency** and the **return value rate**,
and calling either one "the return rate" without saying which is how two
dashboards end up disagreeing.

The rule this recaps, from day 4 worksheet 03:

> **Store the numerator and the denominator. Derive the ratio at the grain the
> question needs.**

Keep `ReturnedSales` and `Sales` in the table — both additive — and let every
consumer divide. A rate stored in a fact table is correct at exactly one grain
and silently wrong at every other, and nothing about the column says which grain
that was.

PART C — small denominators

### Question 5

Rank `Region` x `CustomerSegment` groups by return rate, unfiltered. Print the top five with their row counts, then the distribution of group sizes.
> **NOTE:** day 4 worksheet 03's top-ranked group had **n = 1**. Check before reporting.

In [ ]:
g = sales.groupby(["Region", "CustomerSegment"]).agg(
    n=("LineID", "size"),
    returned=("ReturnedSales", "sum"),
    total=("Sales", "sum"))
g["rate"] = g.returned / g.total

print("groups:", len(g))
print()
print("top 5 by return rate, UNFILTERED:")
print(g.sort_values("rate", ascending=False)[["n", "rate"]].head(5).round(4).to_string())
print()
print("group sizes:")
print(g.n.describe()[["min", "25%", "50%", "75%", "max"]].to_string())
print()
print("groups with fewer than 30 lines:", int((g.n < 30).sum()), "of", len(g))

```
groups: 32

top 5 by return rate, UNFILTERED:
                                         n    rate
Region                CustomerSegment
Northwest Territories Small Business    80  0.3619
                      Consumer          32  0.2650
                      Home Office       87  0.2564
Ontario               Small Business   329  0.2370
                      Corporate        603  0.1693

group sizes:
min      9.00
25%     93.75
50%    217.00
75%    372.50
max    672.00

groups with fewer than 30 lines: 3 of 32
```

**Three of the top five groups are Northwest Territories**, a region that is
4.71% of the business (question 2) — and one of them has just **32 lines**.

This is milder than day 4 worksheet 03, where the top-ranked group had **n = 1**
and a rate of 0.5714. Here the smallest group is 9 and only 3 of 32 fall under
30 lines, so the ranking is not pure noise. But the pattern is identical: **the
top of a ratio ranking is populated by the smallest groups**, because small
denominators produce extreme values in both directions.

The top rate of 0.3619 is over 80 lines. It may well be real — but you cannot
tell it apart from noise without the denominator beside it, which is exactly why
the denominator must be beside it.

Two things to do, always:

**Show `n` next to every rate.** One column, and it converts a misleading ranking
into an honest one. Notice how differently this table reads with the `n` column
than it would without.

**Set a minimum group size before ranking, and say what it is.** Question 6 does
that. The threshold is a judgement call to agree with the business — what is not
acceptable is publishing the unfiltered ranking, because its top is guaranteed to
be the least reliable groups.

### Question 6

Re-rank with a minimum group size of 100 lines. Print the top five and say how many groups survived the filter.

In [ ]:
g = sales.groupby(["Region", "CustomerSegment"]).agg(
    n=("LineID", "size"),
    returned=("ReturnedSales", "sum"),
    total=("Sales", "sum"))
g["rate"] = g.returned / g.total
big = g[g.n >= 100]

print("groups with 100+ lines:", len(big), "of", len(g))
print()
print(big.sort_values("rate", ascending=False)[["n", "rate"]].head(5).round(4).to_string())
print()
print("overall rate: %.4f" % (sales.ReturnedSales.sum() / sales.Sales.sum()))

```
groups with 100+ lines: 23 of 32

                           n    rate
Region  CustomerSegment
Ontario Small Business   329  0.2370
        Corporate        603  0.1693
Prarie  Small Business   258  0.1667
West    Small Business   408  0.1469
Yukon   Home Office      122  0.1466

overall rate: 0.1095
```

A completely different answer, and a usable one.

**Northwest Territories has vanished from the top five entirely** — all three of
its groups had fewer than 100 lines. In its place: Ontario Small Business at
0.2370 over **329 lines**, which is a finding you could act on.

And now a pattern is visible that the unfiltered ranking hid. **`Small Business`
occupies three of the top five**, across three different regions — Ontario,
Prairie and West — at 0.2370, 0.1667 and 0.1469 against an overall rate of
0.1095. Three independent regions, same direction, all well above average.

That consistency is what makes it a signal rather than an artefact. One
anomalous group is a story; the same segment appearing three times across
unrelated regions is a property of how that segment buys or is sold to.

It also matches question 3, where Small Business had the highest segment-level
return rate by revenue (0.1628). Two different cuts of the data agreeing is worth
more than either alone.

**23 of 32 groups survived** the filter, so the threshold discarded 28% of the
groups — and every one of those had fewer than 100 lines. That number belongs in
the footnote of any chart built from this.

PART D — floats

### Question 7

Day 1 worksheet 12 found the same column summing to `1605576.2175` one way and `1605576.2174999998` another. Test whether that happens **here**: sum `Sales` five ways — pandas, Python's `sum`, sorted ascending and descending, and `math.fsum` — and compare them all with `==`.
> **NOTE:** `math.fsum` is exactly rounded, so it is the reference. Predict the result before running it, then read the answer carefully — it is not the obvious one.

In [ ]:
import math
v = sales.Sales
methods = {
    "pandas .sum()":     float(v.sum()),
    "python sum()":      sum(v),
    "sorted ascending":  sum(sorted(v)),
    "sorted descending": sum(sorted(v, reverse=True)),
    "math.fsum (exact)": math.fsum(v),
}
for name, val in methods.items():
    print("  %-20s %r" % (name, val))
print()
vals = set(methods.values())
print("all identical:", len(vals) == 1)
print("distinct results:", len(vals))
print("max difference: %r" % (max(methods.values()) - min(methods.values())))
print()
print("every method agrees with the exactly-rounded fsum:",
      all(x == methods["math.fsum (exact)"] for x in methods.values()))

```
  pandas .sum()        13570810.6285
  python sum()         13570810.6285
  sorted ascending     13570810.6285
  sorted descending    13570810.6285
  math.fsum (exact)    13570810.6285

all identical: True
distinct results: 1
max difference: 0.0

every method agrees with the exactly-rounded fsum: True
```

**They all agree, exactly** — including `math.fsum`, which is guaranteed
exactly rounded.

That is not the answer the question set you up for, and it is the more useful
one. Day 1 worksheet 12 summed *its* column two ways and got `1605576.2175`
against `1605576.2174999998`. Same operation, same language, different data,
different answer.

So the lesson is not "float addition is unreliable, look". It is:

> **Whether reordering changes a float sum is a property of the data, not of the
> code.** You cannot test your way to safety, because the test passes on the
> dataset you have and says nothing about tomorrow's.

That is precisely why `==` on floats is banned as a rule rather than as a
judgement call. A rule you follow when it currently matters is a rule you have
already broken — the failure arrives with new data, in production, in a
reconciliation that has passed every day for a year.

Why it happens to be exact here: these values have at most four decimal places
and the magnitudes are similar, so pandas' pairwise summation loses nothing. Add
one value nine orders of magnitude larger, or accumulate a few million rows, and
it stops being true with no warning.

**Use `math.isclose()`, or compare rounded values, or use `Decimal` for money.
Never `==`.**

### Question 8

Show the same thing costs money. Round `Sales` to 2dp *first*, then sum, and compare against summing then rounding. Print both and the difference.
> **NOTE:** round for presentation, never for storage. Day 2 worksheet 01 found a penny; check what it is here.

In [ ]:
sum_then_round = round(sales.Sales.sum(), 2)
round_then_sum = round(sales.Sales.round(2).sum(), 2)
print("sum, then round: %14.2f" % sum_then_round)
print("round, then sum: %14.2f" % round_then_sum)
print("difference:      %14.2f" % (sum_then_round - round_then_sum))
print()
per_region = sales.groupby("Region").Sales
a = per_region.sum().round(2).sum()
b = round(sales.Sales.sum(), 2)
print("sum of rounded regional totals: %14.2f" % a)
print("rounded grand total:            %14.2f" % b)
print("difference:                     %14.2f" % (a - b))

```
sum, then round:    13570810.63
round, then sum:    13570810.71
difference:               -0.08

sum of rounded regional totals:    13570810.61
rounded grand total:               13570810.63
difference:                              -0.02
```

**Eight cents**, from doing the same two operations in the other order.

Question 7 showed the floats themselves are exact here. This is a different
mechanism entirely and it is fully deterministic: rounding 8,060 values to 2dp
discards a fraction of a cent from each, and those fractions accumulate. Nothing
about floating point is involved — the same thing happens with `Decimal`.

The second pair is the version you will actually meet: **the sum of the rounded
regional totals is 2 cents off the rounded grand total.** That is a report where
the rows do not add up to the footer, and somebody will notice, and the answer
will not be obvious.

Day 2 worksheet 01 found a penny doing this, and worksheet 02 found `Prarie` and
`West` each `-0.01` out while the other six regions reconciled exactly — which is
worse, because six correct rows make the two wrong ones look like a bug rather
than a rounding artefact.

**The rule: round for presentation, never for storage.**

Keep full precision in the table. Round once, at the last moment, in the thing a
human reads. If the rows must add to the footer exactly, compute the footer from
the *unrounded* values and accept that it may differ from the sum of the
displayed rows by a cent — or use a largest-remainder allocation if the
discrepancy is unacceptable.

What you must not do is round on the way in and then wonder why the reconciliation
is 8 cents out.

PART E — say which measure is which

### Question 9

Classify every numeric column in `sales` as `additive`, `non-additive`, or `key/not a measure`, and prove one classification by re-aggregating it to a coarser grain and comparing totals.
> **NOTE:** day 4 worksheet 03 — additivity is a property of a measure *and a grain*, not of a column.

In [ ]:
CLASS = {
    "LineID": "key", "OrderID": "key", "ProductID": "key",
    "CustomerID": "key",
    "OrderQuantity": "additive", "Sales": "additive",
    "Profit": "additive", "ShippingCost": "additive",
    "ReturnedSales": "additive",
    "Discount": "NON-additive (a rate)",
    "UnitPrice": "NON-additive (a per-unit price)",
}
for col in sales.columns:
    if pd.api.types.is_numeric_dtype(sales[col]) and col in CLASS:
        print("  %-15s %s" % (col, CLASS[col]))
print()
whole = sales.Sales.sum()
by_region = sales.groupby("Region").Sales.sum().sum()
by_both = sales.groupby(["Region", "CustomerSegment"]).Sales.sum().sum()
print("Sales re-aggregated:")
print("  ungrouped        %14.2f" % whole)
print("  by region        %14.2f" % by_region)
print("  by region+segment%14.2f" % by_both)
print("  all equal:", len({round(whole, 2), round(by_region, 2),
                           round(by_both, 2)}) == 1)
print()
print("SUM(Discount) is %.2f -- a number, and meaningless" % sales.Discount.sum())

```
  LineID          key             OrderQuantity   additive
  OrderID         key             Sales           additive
  ProductID       key             Profit          additive
  CustomerID      key             ShippingCost    additive
                                  ReturnedSales   additive
  Discount        NON-additive (a rate)
  UnitPrice       NON-additive (a per-unit price)

Sales re-aggregated:
  ungrouped           13570810.63
  by region           13570810.63
  by region+segment   13570810.63
  all equal: True

SUM(Discount) is 400.36 -- a number, and meaningless
```

Eleven numeric columns, and only five are safe to sum.

**The proof is the re-aggregation.** `Sales` totals identically ungrouped, by
region, and by region+segment. That is what additive means, and it is a
one-line test you can run on any measure before exposing it to a BI tool — which
will happily let a user drag it onto any axis and sum it there.

**`SUM(Discount)` is 400.36.** It is a real number, produced without error, and
it means nothing at all: it is the sum of ~8,060 fractions between 0 and 1. A
dashboard would display it without complaint. Same for `SUM(UnitPrice)` — the
total of every product's unit price, which is not a quantity of anything.

The four keys are the quiet trap. `LineID`, `OrderID`, `ProductID` and
`CustomerID` are all integers, so `AVG(CustomerID)` returns a number and `SUM`
returns a bigger one. **The type system cannot tell a key from a measure**, which
is why the classification has to be written down by a person and carried with the
model.

That is day 4 worksheet 03's conclusion, and it is the one thing on this sheet
that no amount of checking will discover for you:

> Additivity is a property of a measure **and a grain**, not of a column. Record
> each measure's aggregation rule as part of the model, because neither the
> column type nor the column name carries it.

### Question 10

Finally, assert that the mean of the regional means equals the true overall mean. **This is supposed to fail.** Read the gap and say which figure you would put in front of a stakeholder.

In [ ]:
per_region = sales.groupby("Region").Sales.agg(["size", "mean"])
unweighted = per_region["mean"].mean()
weighted = sales.Sales.mean()

print("regions:", len(per_region))
print("mean of regional means: %10.4f" % unweighted)
print("true mean over rows:    %10.4f" % weighted)
print("gap:                    %10.4f" % (unweighted - weighted))
print()
print("smallest region: %d rows | largest: %d rows"
      % (per_region["size"].min(), per_region["size"].max()))
print()
assert round(unweighted, 2) == round(weighted, 2), (
    "the average of %d regional averages is %.2f, but the true average order "
    "line is %.2f -- the unweighted version gives every region an equal vote"
    % (len(per_region), unweighted, weighted))

```
regions: 8
mean of regional means:  1687.7244
true mean over rows:     1683.7234
gap:                        4.0010

smallest region: 77 rows | largest: 1921 rows

AssertionError: the average of 8 regional averages is 1687.72, but the true
average order line is 1683.72 -- the unweighted version gives every region an
equal vote
```

**Report the weighted figure: 1683.72.**

It is the average of the thing the question is about — an order line — and it
does not change if the business opens a ninth region tomorrow with three orders
in it. The unweighted 1687.72 would move immediately, by an amount determined
entirely by a region that is 0.04% of the business.

The general test: **if adding a tiny group would move your headline number, you
are using the wrong average.**

And if you genuinely want each region to count equally — a fair comparison across
regions of very different sizes is a legitimate thing to want — then say so in
the label. *"Average of regional averages"* is an honest name. *"Average order
value"* is not.

**What this sheet established:**

| | |
|---|---|
| average of averages | **1687.72 vs 1683.72** — and a **25x** size ratio between the largest and smallest region is why |
| rates, two ways | disagree in **both directions**; Small Business **0.1628 vs 0.1154** |
| which is right | ratio of sums — because a per-line rate here is just 1s and 0s, i.e. return *frequency* |
| ranking by a rate | 3 of the top 5 groups are the region that is **4.71%** of the business; one has **32 lines** |
| with a floor of 100 | a different, usable answer — and `Small Business` appears in **3 of 5** rows |
| float ordering | **all five methods identical** here — which is exactly why `==` stays banned |
| rounding | **8 cents** from ordering; **2 cents** between rows and footer |
| additivity | 5 of 11 numeric columns are summable; `SUM(Discount)` is **400.36** and means nothing |

**The three sheets together:**

- **01** — rows that multiply. *Which side of the join did this number come from?*
- **02** — rows that vanish. *Does this aggregate account for every row it was given?*
- **03** — numbers that survive both and are still wrong. *Am I averaging something I should be summing, and over what?*

All three are the same instruction the whole of week 3 kept arriving at: **the
dangerous results are the ones that do not raise**, so the checks have to be
things you decide to run.